In [24]:
import socket
from pylabnet.hardware.lasers.m2_solstis import Driver
import json

address1 = ("192.168.1.222", 57605)
HOST = '192.168.1.102'                 # Symbolic name meaning all available interfaces
PORT = 1024              # Arbitrary non-privileged port

def _build_message(op, params, transmission_id=None):
    """Builds a message to be sent to the laser driver."""
    if transmission_id is None:
        transmission_id = 1
    else:
        transmission_id = transmission_id
    message = {'message': {'transmission_id': [transmission_id], 'op': op, 'parameters': dict(params)}}
    return json.dumps(message)

In [28]:
s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.bind((HOST, PORT))
s.connect(address1)
interface = s.getsockname()[0]
print(f"Interface: {interface}")
    

Interface: 192.168.1.102


In [35]:
message = _build_message(op='start_link', params={'ip_address': interface})
print(message)
s.sendall(message.encode('utf-8'))
reply = s.recv(1024)
reply = json.loads(reply.decode('utf-8'))
print(f"reply:{reply}")

{"message": {"transmission_id": [1], "op": "start_link", "parameters": {"ip_address": "192.168.1.102"}}}
reply:{'message': {'transmission_id': [1], 'op': 'start_link_reply', 'parameters': {'ip_address': '192.168.1.222', 'status': 'ok'}}}


In [42]:
message = {"message":{"transmission_id":[3], "op":"get_status"}}
print(json.dumps(message).encode('utf-8'))
s.sendall(json.dumps(message).encode('utf-8'))
reply = s.recv(2048)
reply = json.loads(reply.decode('utf-8'))
print(f"reply:{reply}")

b'{"message": {"transmission_id": [3], "op": "get_status"}}'
reply:{'message': {'transmission_id': [3], 'op': 'get_status_reply', 'parameters': {'status': [0], 'wavelength': [737.0241], 'temperature': [220.060333], 'temperature_status': 'off', 'etalon_lock': 'off', 'etalon_voltage': [98.564598], 'cavity_lock': 'off', 'resonator_voltage': [98.412704], 'ecd_lock': 'not_fitted', 'ecd_voltage': 'not_fitted', 'output_monitor': [0.014419], 'etalon_pd_dc': [0.005784], 'dither': 'on'}}}


In [43]:
s.close()